# 02 — Data Cleaning and Cycle Validation

**Goals**
- Build `cycle_manifest.csv` (one row per cycle)
- Flag / exclude contaminated or broken cycles
- Document exclusion reasons
- Output clean list of valid cycles


In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Build the cycle manifest from the raw files
rows = []
for f in sorted(RAW_DIR.glob('*.csv')):
    df = pd.read_csv(f, parse_dates=['timestamp'])
    cid  = df['cycle_id'].iloc[0]
    cond = df['condition'].iloc[0]
    start = df['timestamp'].min()
    end   = df['timestamp'].max()
    team  = 'Alice' if '01' in cid else 'Bob'   # dummy assignment
    notes = 'Clean dummy cycle; no contamination observed'
    rows.append({
        'cycle_id': cid,
        'condition': cond,
        'team_member': team,
        'start_time': start,
        'end_time': end,
        'n_rows': len(df),
        'notes': notes,
        'valid': True,
        'exclusion_reason': ''
    })

manifest = pd.DataFrame(rows)
manifest


In [ ]:
# In a real project you would flag bad cycles here, e.g.:
# manifest.loc[manifest.cycle_id == 'outdoor_cycleXX', 'valid'] = False
# manifest.loc[manifest.cycle_id == 'outdoor_cycleXX', 'exclusion_reason'] = 'rain entered sheltered pot'

excluded = manifest[~manifest['valid']]
if len(excluded) == 0:
    print('No cycles excluded. All cycles are valid.')
else:
    print('Excluded cycles:')
    display(excluded[['cycle_id', 'exclusion_reason']])

valid_cycles = manifest[manifest['valid']]['cycle_id'].tolist()
print('\nValid cycles to proceed with:', valid_cycles)


In [ ]:
# Persist the manifest
manifest.to_csv(PROCESSED_DIR / 'cycle_manifest.csv', index=False)
print('Saved → data/processed/cycle_manifest.csv')
